# Spare-Parts Demand Planning: Planner Override & Learning Experiment

This notebook demonstrates the end-to-end experiment pipeline:
1. Loading synthetic spare-parts demand data (8,500 records)
2. Data cleaning & preprocessing
3. Baseline forecast generation & evaluation
4. Planner override application & role authorization
5. Outcome measurement & baseline vs override error comparison
6. Override learning & reason code ranking
7. Multi-objective trade-off analysis (Cost vs Reliability)
8. Deep error analysis

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Ensure project root is in path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_generator import generate_spare_parts_data
from src.preprocessing import clean_data
from src.forecasting import calculate_forecast_metrics
from src.evaluation import evaluate_demand_outcomes, verify_prototype_targets, analyze_error_patterns
from src.learning import analyze_override_learning
from src.constraint_engine import evaluate_multi_objective_score

## 1. Data Generation & Cleaning

In [ ]:
# Generate 8,500 records
raw_df = generate_spare_parts_data(num_records=8500, random_seed=42)
df = clean_data(raw_df)
print(f"Dataset successfully loaded with shape: {df.shape}")
df.head()

## 2. Baseline Forecast vs Override Outcome Evaluation

In [ ]:
df_eval, summary = evaluate_demand_outcomes(df)
print("=== Summary Metrics ===")
for k, v in summary.items():
    print(f"{k}: {v}")

## 3. Override Learning & Reason Ranking

In [ ]:
learning_results = analyze_override_learning(df_eval)
reason_df = learning_results["reason_learning"]
print("=== Override Reason Intelligence Table ===")
print(reason_df.to_string())

## 4. Visualizing Success Rate by Reason Code

In [ ]:
fig = px.bar(
    reason_df, 
    x="Override_Reason", 
    y="Success_Rate_Pct", 
    hover_data=["Reason_Description", "Total_Overrides", "Avg_Improvement_Pct"],
    title="Planner Override Success Rate by Reason Code",
    labels={"Success_Rate_Pct": "Success Rate (%)", "Override_Reason": "Reason Code"},
    color="Success_Rate_Pct",
    color_continuous_scale="Viridis"
)
fig.show()

## 5. Multi-Objective Optimization Strategy Comparison

In [ ]:
score_a = evaluate_multi_objective_score(cost=450.0, delivery_time=24.0, emissions=45.0, reliability_score=85.0, strategy="A")
score_b = evaluate_multi_objective_score(cost=450.0, delivery_time=24.0, emissions=45.0, reliability_score=85.0, strategy="B")
print("Cost-focused score:", score_a)
print("Reliability-focused score:", score_b)

## 6. Deep Error Analysis

In [ ]:
error_analysis = analyze_error_patterns(df_eval)
print("Top 5 Worst Performing Parts:")
print(error_analysis["worst_parts"])
print("\nDemand Spike Events Count:", error_analysis["spike_events_count"])
print("Stockout Events Count:", error_analysis["stockout_events_count"])